2. Projektni Zadatak: Razvoj Klasifikatora za Analizu
Sentimenta u PySpark-u

In [1]:
from pyspark.sql import SparkSession

sc = SparkSession.builder.appName("FinancialSentimentAnalysis").getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 17:22:14 WARN Utils: Your hostname, Vescos-Mac-mini.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.4 instead (on interface en1)
26/05/03 17:22:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 17:22:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
data = sc.read.csv("data.csv", header=True, inferSchema=True)
data.show()

+--------------------+---------+
|            Sentence|Sentiment|
+--------------------+---------+
|The GeoSolutions ...| positive|
|$ESI on lows, dow...| negative|
|For the last quar...| positive|
|According to the ...|  neutral|
|The Swedish buyou...|  neutral|
|$SPY wouldn't be ...| positive|
|Shell's $70 Billi...| negative|
|SSH COMMUNICATION...| negative|
|Kone 's net sales...| positive|
|The Stockmann dep...|  neutral|
|Circulation reven...| positive|
|$SAP Q1 disappoin...| negative|
|The subdivision m...| positive|
|Viking Line has c...|  neutral|
|Ahlstrom Corporat...|  neutral|
|$FB gone green on...| positive|
|$MSFT SQL Server ...| positive|
|According to L+ñn...|  neutral|
|The company 's sh...|  neutral|
|Elcoteq SE is lis...|  neutral|
+--------------------+---------+
only showing top 20 rows


In [3]:
data.printSchema()

root
 |-- Sentence: string (nullable = true)
 |-- Sentiment: string (nullable = true)



In [4]:
from pyspark.sql.functions import count, when, col

data.select([count(when(col(c).isNull(), c)).alias(c) for c in data.columns]).show()

+--------+---------+
|Sentence|Sentiment|
+--------+---------+
|       0|        0|
+--------+---------+



In [5]:
data.groupBy("sentiment").count().show()

+--------------------+-----+
|           sentiment|count|
+--------------------+-----+
|            positive| 1852|
|             neutral| 3130|
|            negative|  859|
| the damage is do...|    1|
+--------------------+-----+



In [6]:
valid_labels = ["positive", "neutral", "negative"]

data = data.filter(col("Sentiment").isin(valid_labels))

data.groupBy("Sentiment").count().show()

+---------+-----+
|Sentiment|count|
+---------+-----+
| positive| 1852|
|  neutral| 3130|
| negative|  859|
+---------+-----+



In [7]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(
    inputCol="Sentiment",
    outputCol="label"
)

data = label_indexer.fit(data).transform(data)
data.select("Sentiment", "label").show(10)

+---------+-----+
|Sentiment|label|
+---------+-----+
| positive|  1.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
|  neutral|  0.0|
| positive|  1.0|
| negative|  2.0|
| negative|  2.0|
| positive|  1.0|
|  neutral|  0.0|
+---------+-----+
only showing top 10 rows


In [8]:
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print("Train:", train_data.count())
print("Test:", test_data.count())

Train: 4724
Test: 1117


In [9]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover

tokenizer = Tokenizer(
    inputCol="Sentence",
    outputCol="tokens"
)

stop_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

In [10]:
from nltk.stem import PorterStemmer
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

stemmer = PorterStemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

stem_udf = udf(stem_tokens, ArrayType(StringType()))

In [11]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def evaluate_model(predictions):
    metrics = {}
    
    for metric in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
        evaluator = MulticlassClassificationEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName=metric
        )
        metrics[metric] = evaluator.evaluate(predictions)
    
    return metrics

In [12]:
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier

def get_model(model_name, params=None):
    params = params or {}

    if model_name == "lr":
        return LogisticRegression(featuresCol="features", labelCol="label", **params)
    
    elif model_name == "dt":
        return DecisionTreeClassifier(featuresCol="features", labelCol="label", seed=42, **params)
    
    elif model_name == "rf":
        return RandomForestClassifier(featuresCol="features", labelCol="label", seed=42, **params)
    
    else:
        raise ValueError("Unknown model")

In [13]:
def preprocess_data(df):
    tokenized = tokenizer.transform(df)
    filtered = stop_remover.transform(tokenized)

    stemmed = filtered.withColumn(
        "stemmed_tokens",
        stem_udf(col("filtered_tokens"))
    )

    return stemmed

In [14]:
from pyspark.ml.feature import CountVectorizer, IDF

def build_tfidf_features(train_df, test_df):
    cv = CountVectorizer(
        inputCol="stemmed_tokens",
        outputCol="raw_features",
        vocabSize=5000,
        minDF=5
    )
    cv_model = cv.fit(train_df)

    train_featurized = cv_model.transform(train_df)
    test_featurized = cv_model.transform(test_df)

    idf = IDF(
        inputCol="raw_features",
        outputCol="features"
    )
    idf_model = idf.fit(train_featurized)

    train_final = idf_model.transform(train_featurized)
    test_final = idf_model.transform(test_featurized)

    return train_final, test_final

In [15]:
from pyspark.ml.feature import NGram

def build_bigram_features(train_df, test_df):
    ngram = NGram(
        n=2,
        inputCol="stemmed_tokens",
        outputCol="bigrams"
    )

    train_ngram = ngram.transform(train_df)
    test_ngram = ngram.transform(test_df)

    cv = CountVectorizer(
        inputCol="bigrams",
        outputCol="raw_features",
        vocabSize=5000,
        minDF=5
    )
    cv_model = cv.fit(train_ngram)

    train_featurized = cv_model.transform(train_ngram)
    test_featurized = cv_model.transform(test_ngram)

    idf = IDF(
        inputCol="raw_features",
        outputCol="features"
    )
    idf_model = idf.fit(train_featurized)

    train_final = idf_model.transform(train_featurized)
    test_final = idf_model.transform(test_featurized)

    return train_final, test_final

In [16]:
from pyspark.ml.feature import Word2Vec

def build_word2vec_features(train_df, test_df):
    word2vec = Word2Vec(
        vectorSize=100,
        minCount=1,
        inputCol="stemmed_tokens",
        outputCol="features"
    )

    w2v_model = word2vec.fit(train_df)

    train_final = w2v_model.transform(train_df)
    test_final = w2v_model.transform(test_df)

    return train_final, test_final

In [17]:
def prepare_features(train_data, test_data, feature_method):
    train_processed = preprocess_data(train_data)
    test_processed = preprocess_data(test_data)

    if feature_method == "tfidf":
        return build_tfidf_features(train_processed, test_processed)

    elif feature_method == "bigram_tfidf":
        return build_bigram_features(train_processed, test_processed)

    elif feature_method == "word2vec":
        return build_word2vec_features(train_processed, test_processed)

    else:
        raise ValueError("Unknown feature method")

In [18]:
def train_and_evaluate(train_final, test_final, model_name, params=None):
    model = get_model(model_name, params)
    trained_model = model.fit(train_final)

    predictions = trained_model.transform(test_final)

    metrics = evaluate_model(predictions)

    return {
        "model": trained_model,
        "metrics": metrics
    }

In [19]:
def run_experiment(train_data, test_data, feature_method, model_name, params=None):
    train_final, test_final = prepare_features(train_data, test_data, feature_method)
    result = train_and_evaluate(train_final, test_final, model_name, params)

    return {
        "feature_method": feature_method,
        "model_name": model_name,
        "params": params or {},
        "metrics": result["metrics"],
        "model": result["model"]
    }

In [20]:
param_grid = {
    "lr": [
        {"maxIter": 100, "regParam": 0.01, "elasticNetParam": 0.0},
        {"maxIter": 100, "regParam": 0.1,  "elasticNetParam": 0.0},
        {"maxIter": 100, "regParam": 0.01, "elasticNetParam": 0.5},
        {"maxIter": 100, "regParam": 0.1,  "elasticNetParam": 0.5},
        {"maxIter": 100, "regParam": 0.01, "elasticNetParam": 1.0},
    ],

    "dt": [
        {"maxDepth": 8,  "minInstancesPerNode": 2},
        {"maxDepth": 10, "minInstancesPerNode": 2},
        {"maxDepth": 12, "minInstancesPerNode": 5},
        {"maxDepth": 15, "minInstancesPerNode": 10},
    ],

    "rf": [
        {"numTrees": 100, "maxDepth": 10, "featureSubsetStrategy": "sqrt"},
        {"numTrees": 150, "maxDepth": 12, "featureSubsetStrategy": "sqrt"},
    ]
}

In [21]:
def run_all_experiments(train_data, test_data, param_grid):
    results = {}

    feature_methods = ["tfidf", "bigram_tfidf", "word2vec"]
    prepared_data = {}

    for feature_method in feature_methods:
        print(f"Preparing {feature_method} features...")
        train_final, test_final = prepare_features(train_data, test_data, feature_method)
        # cache
        train_final = train_final.cache()
        test_final = test_final.cache()
        # force computation
        train_final.count()
        test_final.count()
        prepared_data[feature_method] = (train_final, test_final)

    for feature_method, (train_final, test_final) in prepared_data.items():
        for model_name, param_list in param_grid.items():
            for params in param_list:
                key = f"{feature_method}_{model_name}_{params}"

                print(f"Running {key}...")

                result = train_and_evaluate(
                    train_final,
                    test_final,
                    model_name,
                    params
                )

                results[key] = result["metrics"]

    return results

In [22]:
all_results = run_all_experiments(train_data, test_data, param_grid)

Preparing tfidf features...


Preparing bigram_tfidf features...
Preparing word2vec features...


26/05/03 17:22:24 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Running tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.0}...
Running tfidf_lr_{'maxIter': 100, 'regParam': 0.1, 'elasticNetParam': 0.0}...
Running tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.5}...
Running tfidf_lr_{'maxIter': 100, 'regParam': 0.1, 'elasticNetParam': 0.5}...
Running tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 1.0}...
Running tfidf_dt_{'maxDepth': 8, 'minInstancesPerNode': 2}...
Running tfidf_dt_{'maxDepth': 10, 'minInstancesPerNode': 2}...
Running tfidf_dt_{'maxDepth': 12, 'minInstancesPerNode': 5}...
Running tfidf_dt_{'maxDepth': 15, 'minInstancesPerNode': 10}...
Running tfidf_rf_{'numTrees': 100, 'maxDepth': 10, 'featureSubsetStrategy': 'sqrt'}...


26/05/03 17:22:33 WARN DAGScheduler: Broadcasting large task binary with size 1113.8 KiB
26/05/03 17:22:33 WARN DAGScheduler: Broadcasting large task binary with size 1338.2 KiB
26/05/03 17:22:33 WARN DAGScheduler: Broadcasting large task binary with size 1573.6 KiB
26/05/03 17:22:34 WARN DAGScheduler: Broadcasting large task binary with size 1083.1 KiB
26/05/03 17:22:34 WARN DAGScheduler: Broadcasting large task binary with size 1083.1 KiB
26/05/03 17:22:34 WARN DAGScheduler: Broadcasting large task binary with size 1083.1 KiB
26/05/03 17:22:34 WARN DAGScheduler: Broadcasting large task binary with size 1083.1 KiB


...

26/05/03 17:22:46 WARN DAGScheduler: Broadcasting large task binary with size 1093.9 KiB
26/05/03 17:22:46 WARN DAGScheduler: Broadcasting large task binary with size 1254.1 KiB
26/05/03 17:22:46 WARN DAGScheduler: Broadcasting large task binary with size 1418.1 KiB
26/05/03 17:22:46 WARN DAGScheduler: Broadcasting large task binary with size 1574.5 KiB
26/05/03 17:22:47 WARN DAGScheduler: Broadcasting large task binary with size 1197.7 KiB
26/05/03 17:22:47 WARN DAGScheduler: Broadcasting large task binary with size 1197.7 KiB
26/05/03 17:22:47 WARN DAGScheduler: Broadcasting large task binary with size 1197.7 KiB
26/05/03 17:22:47 WARN DAGScheduler: Broadcasting large task binary with size 1197.7 KiB


Running word2vec_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.0}...
Running word2vec_lr_{'maxIter': 100, 'regParam': 0.1, 'elasticNetParam': 0.0}...
Running word2vec_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.5}...
Running word2vec_lr_{'maxIter': 100, 'regParam': 0.1, 'elasticNetParam': 0.5}...
Running word2vec_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 1.0}...
Running word2vec_dt_{'maxDepth': 8, 'minInstancesPerNode': 2}...
Running word2vec_dt_{'maxDepth': 10, 'minInstancesPerNode': 2}...
Running word2vec_dt_{'maxDepth': 12, 'minInstancesPerNode': 5}...
Running word2vec_dt_{'maxDepth': 15, 'minInstancesPerNode': 10}...
Running word2vec_rf_{'numTrees': 100, 'maxDepth': 10, 'featureSubsetStrategy': 'sqrt'}...


26/05/03 17:22:55 WARN DAGScheduler: Broadcasting large task binary with size 1127.6 KiB
26/05/03 17:22:55 WARN DAGScheduler: Broadcasting large task binary with size 1923.2 KiB
26/05/03 17:22:56 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/05/03 17:22:56 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/05/03 17:22:57 WARN DAGScheduler: Broadcasting large task binary with size 6.5 MiB
26/05/03 17:22:58 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/05/03 17:22:58 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/05/03 17:22:58 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/05/03 17:22:58 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB


Running word2vec_rf_{'numTrees': 150, 'maxDepth': 12, 'featureSubsetStrategy': 'sqrt'}...


26/05/03 17:22:59 WARN DAGScheduler: Broadcasting large task binary with size 1622.8 KiB
26/05/03 17:23:00 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/05/03 17:23:01 WARN DAGScheduler: Broadcasting large task binary with size 4.6 MiB
26/05/03 17:23:01 WARN DAGScheduler: Broadcasting large task binary with size 6.9 MiB
26/05/03 17:23:02 WARN DAGScheduler: Broadcasting large task binary with size 1100.1 KiB
26/05/03 17:23:02 WARN DAGScheduler: Broadcasting large task binary with size 9.8 MiB
26/05/03 17:23:03 WARN DAGScheduler: Broadcasting large task binary with size 1276.9 KiB
26/05/03 17:23:04 WARN DAGScheduler: Broadcasting large task binary with size 13.0 MiB
26/05/03 17:23:04 WARN DAGScheduler: Broadcasting large task binary with size 1387.3 KiB
26/05/03 17:23:05 WARN DAGScheduler: Broadcasting large task binary with size 16.3 MiB
26/05/03 17:23:05 WARN DAGScheduler: Broadcasting large task binary with size 1428.2 KiB
26/05/03 17:23:07 WARN DAGScheduler: 

In [23]:
best_model = max(all_results.items(), key=lambda x: x[1]["f1"])

print("Best configuration:")
print(best_model[0])
print(best_model[1])

Best configuration:
tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.5}
{'accuracy': 0.7332139659803044, 'f1': 0.7170727124670712, 'weightedPrecision': 0.7132784739583369, 'weightedRecall': 0.7332139659803044}


In [24]:
import pandas as pd

results_table = []

for name, metrics in all_results.items():
    row = {"experiment": name}
    row.update(metrics)
    results_table.append(row)

results_df = pd.DataFrame(results_table)
results_df.sort_values(by="f1", ascending=False)

,experiment,accuracy,f1,weightedPrecision,weightedRecall
2,"tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'e...",0.733214,0.717073,0.713278,0.733214
4,"tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'e...",0.730528,0.707126,0.713127,0.730528
8,"tfidf_dt_{'maxDepth': 15, 'minInstancesPerNode...",0.710833,0.678756,0.683299,0.710833
1,"tfidf_lr_{'maxIter': 100, 'regParam': 0.1, 'el...",0.677708,0.668560,0.662851,0.677708
6,"tfidf_dt_{'maxDepth': 10, 'minInstancesPerNode...",0.710833,0.662838,0.694316,0.710833
7,"tfidf_dt_{'maxDepth': 12, 'minInstancesPerNode...",0.697404,0.660174,0.658338,0.697404
5,"tfidf_dt_{'maxDepth': 8, 'minInstancesPerNode'...",0.709042,0.659063,0.718413,0.709042
0,"tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'e...",0.614145,0.617059,0.620192,0.614145
22,"word2vec_lr_{'maxIter': 100, 'regParam': 0.01,...",0.643688,0.599664,0.600935,0.643688
3,"tfidf_lr_{'maxIter': 100, 'regParam': 0.1, 'el...",0.668756,0.598519,0.568344,0.668756


Interpretacija rezultata

Najbolji model na Financial Sentiment Analysis datasetu je Logistic Regression sa TF-IDF reprezentacijom - tfidf_lr_{'maxIter': 100, 'regParam': 0.01, 'elasticNetParam': 0.5}, koji je ostvario najbolje rezultate (Accuracy ≈ 0.733, F1 ≈ 0.717).

TF-IDF se pokazao boljim od Word2Vec i bigrama, što znači da su ključne pojedinačne reči dovoljne za dobru klasifikaciju sentimenta.

Decision Tree je dao solidne, ali slabije rezultate, dok Random Forest i bigram pristupi imaju najlošije performanse zbog visoke dimenzionalnosti i šuma.

Zaključak

Najefikasniji pristup je TF-IDF + Logistic Regression, jer daje najbolji balans tačnosti i stabilnosti, što potvrđuje da su obrasci u podacima uglavnom linearno separabilni.